# talk

> A walk as a talk for a group: the deck shape, its checks, a talk made from a walk, the run log and the recap.

A presenter can give a walk to a group. The talk is a list of slides. Each slide shows a part of the walk, one beat at a time,
and it can have one question for the room. This module is the engine of a talk: the shape of a deck, the checks of that shape,
the questions, a talk made from a walk, the store calls,
the public deck that the room sees, the run log, and the recap after the talk. It imports no web code, so Leela can use the same shape and checks.

In [ ]:
#| default_exp talk

In [ ]:
#| hide
import os, tempfile
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings

In [ ]:
#| hide
from fastcore.test import *

The imports are light: the standard library, the store, the journey, the demos, the model calls, the prompts and the settings.

In [ ]:
#| export
import re, json, math, time, uuid, secrets, hashlib, base64, html
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from typing import Callable
from fastcore.utils import *
from abhyasa.store import Learning, slugify, _src
from abhyasa.journey import minutes
from abhyasa.llm import ask_json, for_task
from abhyasa.prompts import register, prompt
from abhyasa.settings import val
import abhyasa.demo  # adds Learning.demo_for

## What a talk is

A talk is a document on the learning shelf, of kind `talk`. It points at its walk in `src` and copies what it shows, so a change
to the talk never changes the walk. `minutes` is the slot of the presenter. `phones` is the choice "Room phones". `live` is the id
of the run that is open now, or None.

These names give the shape. A slide has one of `SLIDE_KINDS`, and a question has one of `ASK_KINDS`.

In [ ]:
#| export
TALK_SCHEMA = 1
SLIDE_KINDS = ('idea', 'pause', 'questions', 'recap', 'primer', 'board', 'timeline', 'code')
ASK_KINDS = ('choice', 'order', 'words', 'scale', 'none')
BEATS = ('passage', 'figure', 'demo', 'ask')
QUESTIONS_MIN = 5   # the minutes of a `questions` slide
RECAP_MIN = 2       # the minutes of a `recap` slide
WORDS_MOST = 80     # the most characters of a words answer

## Slides and beats

Each press of D shows one beat more. `passage`, `figure` and `demo` show a part of the slide. `ask` opens the question.
`lit:<text>` lights a text in the passage, and `lit:#<id>` lights a part of the figure or the demo. Leela adds `lines:a-b` and
`out` for its code slides.

A beat that shows a part needs that part on the slide. This table gives the part for each beat.

In [ ]:
#| export
_PART = dict(passage='passage', figure='figure', demo='demo', ask='question')

def _beat_ok(b) -> bool:
    "True when `b` is a beat that the stage knows."
    return isinstance(b, str) and (b in BEATS or b == 'out' or b.startswith('lit:') or bool(re.fullmatch(r'lines:\d+(-\d+)?', b)))

A figure is an image id, never a path or a URL: `asset:<path in the vault assets>`. An id with another scheme, a `..` part,
a leading `/`, a backslash, a second `:` or a control character is refused.

In [ ]:
#| export
def _image_ok(f, schemes=('asset',)) -> bool:
    "True when `f` is an image id with one of `schemes`."
    if not isinstance(f, str) or ':' not in f or any(ord(ch) < 32 or ord(ch) == 127 for ch in f): return False
    sch, p = f.split(':', 1)
    return sch in schemes and bool(p) and ':' not in p and not p.startswith('/') and '\\' not in p and '..' not in p.split('/')

A plain path and a path out of the assets folder are not image ids. A good id is.

In [ ]:
assert _image_ok('asset:attention/images/fig2.png')
assert not any(_image_ok(f) for f in ('/Users/k/fig.png', 'asset:../vault.db', 'asset:/etc/x.png', 'ws:a.png', 'asset:a:b.png', 'asset:a\\b.png', 'asset:a\nb.png', None))

`_ask_problems` checks the question of one slide. A choice needs 2 to 6 options and the index of one of them. An order
needs the indexes of all its options, each one time.

In [ ]:
#| export
def _ask_problems(sid, a) -> list:
    "The problems of the question `a` of slide `sid`."
    if a is None: return []
    k = a.get('kind') if isinstance(a, dict) else a
    if not isinstance(a, dict) or k not in ASK_KINDS: return [f'{sid}: unknown ask kind {k!r}']
    opts, ans = a.get('options'), a.get('answer')
    is_int = lambda x: isinstance(x, int) and not isinstance(x, bool)
    if k == 'choice':
        if not isinstance(opts, list) or not 2 <= len(opts) <= 6: return [f'{sid}: a choice needs 2 to 6 options']
        if not (is_int(ans) and 0 <= ans < len(opts)): return [f'{sid}: the answer {ans!r} is not an option']
    if k == 'order':
        n = len(opts) if isinstance(opts, list) else 0
        if not (isinstance(ans, list) and all(map(is_int, ans)) and sorted(ans) == list(range(n))): return [f'{sid}: the answer {ans!r} is not an order of the options']
    return []

`_beat_problems` checks the beats of one slide against what the slide shows.

In [ ]:
#| export
def _beat_problems(sid, s:dict, show:dict) -> list:
    "The problems of the beats of slide `s`."
    out, beats = [], s.get('beats') or []
    for b in beats if isinstance(beats, list) else [beats]:
        if not _beat_ok(b): out.append(f'{sid}: unknown beat {b!r}'); continue
        if b in _PART and not (s.get('ask') if b == 'ask' else show.get(b)): out.append(f'{sid}: the beat {b!r} needs a {_PART[b]} to show')
        if b.startswith('lit:') and not b.startswith('lit:#'):
            text, passage = b[4:], show.get('passage')
            if not (text and isinstance(passage, str) and text in passage): out.append(f'{sid}: the beat {b!r} lights text that is not in the passage')
    return out

`check_talk` gives the problems of a talk, slide by slide in slide order. One slide can give more than one problem. A good talk
gives `[]`. It never raises, so a host can show the problems of a talk from a file.

In [ ]:
#| export
def check_talk(t:dict, schemes=('asset',)) -> list:
    "The problems of talk `t`; `[]` when the talk is good. It never raises."
    if not isinstance(t, dict): return ['the talk is not a dict']
    out = [] if (v := t.get('schema')) == TALK_SCHEMA else [f'schema {v} is not 1']
    slides, seen = t.get('slides'), set()
    if not isinstance(slides, list): return out + ['the slides are not a list']
    for s in slides:
        if not isinstance(s, dict): out.append('a slide is not a dict'); continue
        sid = s.get('id')
        try: twice = sid in seen; seen.add(sid)
        except TypeError: twice = False
        if twice: out.append(f'{sid}: the slide id is used twice')
        if (k := s.get('kind')) not in SLIDE_KINDS: out.append(f'{sid}: unknown kind {k!r}')
        out += _ask_problems(sid, s.get('ask'))
        show = s.get('show') if isinstance(s.get('show'), dict) else {}
        out += _beat_problems(sid, s, show)
        if (f := show.get('figure')) is not None and not _image_ok(f, schemes): out.append(f'{sid}: the figure {f!r} is not an image id')
    return out

`TALK_CASES` holds a good talk and one changed copy of it for each problem, with the problems that each must give. Leela runs
the same cases on its own copy of the checks.

In [ ]:
#| export
def _good() -> dict:
    "A small good talk with one slide."
    return dict(id='t-1', kind='talk', schema=1, title='T', src={'walk': 'w-1'}, minutes=10, phones=False, live=None,
                slides=[dict(id='s1', kind='idea', title='A', section='', notes='', mins=3,
                             show=dict(passage='Scores become weights.', figure='asset:a/fig.png', demo=None),
                             beats=['passage', 'lit:weights', 'figure'], ask=dict(kind='choice', q='Q?', options=['a', 'b'], answer=1))])

def _case(talk:dict=None, **slide) -> dict:
    "The good talk with the keys of `talk` and the keys of `slide` on its first slide changed."
    t = _good(); t.update(talk or {}); t['slides'][0].update(slide)
    return t

def _with_show(**kw): return {**_good()['slides'][0]['show'], **kw}
def _with_ask(**kw): return {**_good()['slides'][0]['ask'], **kw}

TALK_CASES = [
    (_good(), []),
    (_case(dict(schema=2)), ['schema 2 is not 1']),
    (_case(kind='video'), ["s1: unknown kind 'video'"]),
    (_case(ask=_with_ask(answer=2)), ['s1: the answer 2 is not an option']),
    (_case(ask=_with_ask(options=['a'], answer=0)), ['s1: a choice needs 2 to 6 options']),
    (_case(beats=['passage', 'demo']), ["s1: the beat 'demo' needs a demo to show"]),
    (_case(beats=['passage', 'lit:gradients']), ["s1: the beat 'lit:gradients' lights text that is not in the passage"]),
    (_case(show=_with_show(figure='/Users/k/fig.png')), ["s1: the figure '/Users/k/fig.png' is not an image id"]),
    (_case(show=_with_show(figure='asset:../vault.db')), ["s1: the figure 'asset:../vault.db' is not an image id"]),
    (dict(_good(), slides=_good()['slides'] * 2), ['s1: the slide id is used twice']),
    (_case(ask=_with_ask(kind='poll')), ["s1: unknown ask kind 'poll'"]),
    (_case(beats=['passage', 'fly']), ["s1: unknown beat 'fly'"]),
    (_case(ask=dict(kind='order', q='Q?', options=['a', 'b', 'c'], answer=[0, 0, 1])), ['s1: the answer [0, 0, 1] is not an order of the options'])]

A talk that is not a dict, or a slide with odd values, gives problems and does not raise.

In [ ]:
assert check_talk(None) == ['the talk is not a dict'] and check_talk(dict(schema=1)) == ['the slides are not a list']
assert check_talk(dict(schema=1, slides=[dict(id=['x'], kind='idea', beats='fly', ask='poll', show=None)])) == \
    ["['x']: unknown ask kind 'poll'", "['x']: unknown beat 'fly'"]

## Questions for the room

A demo gate with fixed options can be a room question with no model call. `ask_from_gate` has the contract of the future
`rupaka.poll.of_gate`. The options are a list, or `{"from": path}` in the state of the spec. The answer is `{"is": value}`, or a
path in the state. A path that is not in the state (it needs `derive`) gives None, and so do fewer than 2 or more than 6 options.
The prompt is drawn with the values of the state. A `{name}` with no value stays as it is.

In [ ]:
#| export
_NONE = object()

def _at(state, path:str):
    "The value at the dotted `path` in `state`, or `_NONE`."
    v = state
    for k in str(path).split('.'):
        if isinstance(v, dict) and k in v: v = v[k]
        elif isinstance(v, list) and k.isdigit() and int(k) < len(v): v = v[int(k)]
        else: return _NONE
    return v

def _gates(bs):
    "The gate blocks in `bs`, nested blocks too, in order."
    for b in bs or []:
        if not isinstance(b, dict): continue
        if b.get('type') == 'gate': yield b
        yield from _gates(b.get('blocks'))

`ask_from_gate` gives a choice from gate `gate` of `spec`, or None.

In [ ]:
#| export
def ask_from_gate(spec:dict, gate:str) -> dict|None:
    "A choice question from the gate `gate` of `spec`, with the index of the right option; None when the gate can not be a room question."
    g, state = first(b for b in _gates((spec or {}).get('blocks')) if b.get('id') == gate), (spec or {}).get('state') or {}
    if g is None: return None
    opts = g.get('options')
    if isinstance(opts, dict): opts = _at(state, opts.get('from', ''))
    ans = g.get('answer')
    ans = ans['is'] if isinstance(ans, dict) and 'is' in ans else _at(state, ans) if isinstance(ans, str) else _NONE
    if not isinstance(opts, list) or not 2 <= len(opts) <= 6 or ans is _NONE or ans not in opts: return None
    q = re.sub(r'\{(\w+)\}', lambda m: str(state[m[1]]) if m[1] in state else m[0], g.get('prompt') or '')
    return dict(kind='choice', q=q, options=list(opts), answer=opts.index(ans)) if q else None

`public_ask` keeps only what the room can see: the kind, the question, the options, and the most characters of a words answer.

In [ ]:
#| export
def public_ask(ask:dict|None) -> dict|None:
    "The part of `ask` that the room sees: no answer and no source."
    return None if ask is None else {k: ask[k] for k in ('kind', 'q', 'options', 'most') if k in ask}

A short check becomes a choice of 4 options through one model call, with the prompt `talk.choice`. The answer stays in the
talk. When the call fails, or the reply is bad, the question is a words question.

In [ ]:
#| export
CHOICE_SP = register('talk.choice', 'Talk question', 'How the tutor turns a short check into a choice question for a room.', """You turn one check into a choice question. A presenter shows the question to a room, and each person picks one option.
- Ask about the idea that the statement and the passage give. Ask one thing only.
- Write a short question that a person can read in a few seconds.
- Write 4 short options. One option is right. Each other option is a mistake that a learner can make.
- Do not use "all of the above" or "none of the above". Do not put the answer in the question.
Return JSON: {"q": "the question", "options": ["4 options"], "answer": the index of the right option, from 0}""")

`_choice` gives the reply as a choice when it has a question, 4 different options and the index of one of them, else None.

In [ ]:
#| export
def _choice(r) -> dict|None:
    "The reply `r` as a choice question, or None when it is bad."
    if not isinstance(r, dict): return None
    q, opts, ans = r.get('q'), r.get('options'), r.get('answer')
    good = (isinstance(q, str) and q.strip() and isinstance(opts, list) and len(opts) == 4 and all(isinstance(o, str) and o.strip() for o in opts)
            and len(set(opts)) == 4 and isinstance(ans, int) and not isinstance(ans, bool) and 0 <= ans < 4)
    return dict(kind='choice', q=q.strip(), options=[o.strip() for o in opts], answer=ans) if good else None

`ask_from_check` makes the call with the model task `talk`. `ask_for_concept` does the same with the statement of the idea as the passage.

In [ ]:
#| export
def ask_from_check(lr:Learning, ask, slug:str, passage:str='') -> dict:
    "A choice question on concept `slug` from one model call, or a words question when the call fails."
    c = lr.concept(slug) or AttrDict(name=slug, statement='')
    p = f"Idea: {c.get('name') or slug}\nStatement: {c.get('statement') or ''}\nPassage: {passage}"
    try: r = _choice(ask_json(for_task(ask, 'talk'), p, prompt('talk.choice')))
    except Exception: r = None
    return r or dict(kind='words', q=f"Say in one or two sentences: what is {c.get('name') or slug}?", most=WORDS_MOST)

def ask_for_concept(lr:Learning, ask, slug:str) -> dict:
    "`ask_from_check` with the statement of concept `slug` as the passage."
    return ask_from_check(lr, ask, slug, (lr.concept(slug) or {}).get('statement') or '')

## Minutes

A slide with a question takes the minutes of its walk step plus `talks.ask_min`. The talk takes the sum of its slides.

In [ ]:
#| export
def slide_minutes(step:dict, ask:dict|None) -> int:
    "The minutes of the slide of walk step `step` with the question `ask`."
    return minutes(step) + (val('talks.ask_min') if ask and ask.get('kind') != 'none' else 0)

def talk_minutes(t) -> int:
    "The minutes of all the slides of talk `t`."
    return sum(s.get('mins') or 0 for s in t['slides'])

## A talk from a walk

Each walk step that is not skipped gives one `idea` slide. A demo or an order check takes its question from the first gate of the
demo that can be a room question. A step with no demo, or with no such gate, uses its passage and a short check. A code check
gives a words question, because a talk never runs code.

`_idea_ask` gives the demo slug (or None) and the question of one step.

In [ ]:
#| export
def _idea_ask(lr:Learning, ask, s:dict, passage:str, choice:bool) -> tuple:
    "The demo slug and the question of walk step `s`."
    kind = (s.get('check') or {}).get('kind') or 'short'
    if kind in ('demo', 'order') and (d := lr.demo_for(s['slug'], 'order' if kind == 'order' else None)):
        for g in _gates(d.spec.get('blocks')):
            if (a := ask_from_gate(d.spec, g.get('id'))): return d.slug, dict(a, src=dict(demo=d.slug, gate=g['id']))
    c = lr.concept(s['slug']) or {}
    if kind == 'code' or not choice: a = dict(kind='words', q=f"Say in one or two sentences: what is {c.get('name') or s['slug']}?", most=WORDS_MOST)
    else: a = ask_from_check(lr, ask, s['slug'], passage)
    return None, dict(a, src=dict(check='code' if kind == 'code' else 'short'))

`_idea` makes the slide of one step. Its beats show the passage, then the figure, then the demo, each when the slide has it.

In [ ]:
#| export
def _idea(lr:Learning, ask, i:int, s:dict, sid:str, choice:bool) -> dict:
    "The `idea` slide `sid` of walk step `i`."
    r = s.get('read') or {}
    passage, fig = r.get('text') or '', r.get('figure')
    demo, a = _idea_ask(lr, ask, s, passage, choice)
    show = dict(passage=passage, figure=f'asset:{fig}' if fig else None, demo=demo)
    beats = [b for b in ('passage', 'figure', 'demo') if show[b]]
    c = lr.concept(s['slug']) or {}
    return dict(id=sid, kind='idea', step=i, concept=s['slug'], title=c.get('name') or s['slug'], section=r.get('section') or '',
                show=show, beats=beats, notes=s.get('note') or '', ask=a, mins=slide_minutes(s, a))

`talk_from_walk` makes the slides, then a `questions` slide and a `recap` slide, and saves the talk. The slot is the estimate
rounded up to a multiple of 5 minutes. The id is made from the title, so it matches `^[a-z0-9-]{1,48}$`.

In [ ]:
#| export
def _end(sid:str, kind:str, title:str, mins:int) -> dict:
    "A closing slide with no step and no question."
    return dict(id=sid, kind=kind, title=title, section='', show=dict(passage='', figure=None, demo=None), beats=[], notes='', ask=None, mins=mins)

def _talk_id(title:str) -> str:
    "A new talk id from `title`."
    return re.sub(r'[^a-z0-9-]', '', f'{slugify(title)[:34]}-talk-{uuid.uuid4().hex[:4]}')[:48]

def talk_from_walk(lr:Learning, ask, jid:str, choice:bool|None=None) -> AttrDict:
    "Make a talk from walk `jid` and save it. `choice` (default `talks.choice`) makes a choice question from each short check."
    j = lr.journey(jid)
    if j is None: raise KeyError(f'no journey {jid!r}')
    choice = val('talks.choice') if choice is None else choice
    steps = [(i, s) for i, s in enumerate(j.steps) if s.get('status') != 'skip']
    slides = [_idea(lr, ask, i, s, f's{k+1}', choice) for k, (i, s) in enumerate(steps)]
    n = len(slides)
    slides += [_end(f's{n+1}', 'questions', 'Questions', QUESTIONS_MIN), _end(f's{n+2}', 'recap', 'What we covered', RECAP_MIN)]
    t = dict(id=_talk_id(j.title), kind='talk', schema=TALK_SCHEMA, title=j.title, src={'walk': jid}, made=time.time(),
             minutes=0, phones=False, live=None, slides=slides)
    t['minutes'] = 5 * math.ceil(talk_minutes(t) / 5)
    return lr.put_talk(t)

## Store calls

A talk is a document of kind `talk`. `put_talk` refuses a talk with problems, and replaces an earlier talk with the same id.

In [ ]:
#| export
def _tmeta(r): return AttrDict({**json.loads(r['meta'] or '{}'), 'doc_id': r['id']})

@patch
def put_talk(self:Learning, t:dict) -> AttrDict:
    "Save talk `t`, and replace an earlier one with the same id. A talk with problems raises `ValueError`."
    t = {k: v for k, v in dict(t).items() if k != 'doc_id'}
    if (ps := check_talk(t)): raise ValueError('; '.join(ps))
    t['updated'] = time.time()
    text = f"Talk: {t['title']} ({t['id']})\n\nSlides: {', '.join(s.get('title') or '' for s in t['slides'])}"
    r = self.v.add(text, title=f"Talk: {t['title']}", source=_src('talk', t['id']), kind='talk', meta=t)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **t)
    return self.talk(t['id'])

`talk` gives one talk by id, `talks` gives all of them with the newest first, and `drop_talk` removes one.

In [ ]:
#| export
@patch
def talk(self:Learning, tid:str) -> AttrDict|None:
    "One talk by id."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('talk', tid)]))
    return _tmeta(r) if r else None

@patch
def talks(self:Learning) -> L:
    "All talks, the most recently changed first."
    return L(sorted(map(_tmeta, self._docs('talk')), key=lambda t: -t.get('updated', 0)))

@patch
def drop_talk(self:Learning, tid:str):
    "Remove a talk. Its walk stays."
    if (t := self.talk(tid)): self.v.forget(t.doc_id)

## What the room sees

The room sees a public deck. It is made from a list of allowed fields, never by removing fields, so a new field in a talk does
not reach the room. It has no notes, no answers, no source of a question, no demo and no path. A figure comes by the sha256 of
its bytes.

Only png, jpeg, gif and webp files can be figures. `image_type` finds the type from the first bytes of the file, not from its name.

In [ ]:
#| export
IMAGE_TYPES = {b'\x89PNG\r\n\x1a\n': 'image/png', b'\xff\xd8\xff': 'image/jpeg', b'GIF87a': 'image/gif', b'GIF89a': 'image/gif'}
IMAGE_SUFFIXES = ('.png', '.jpg', '.jpeg', '.gif', '.webp')

def image_type(b:bytes) -> str|None:
    "The image type of the bytes `b`, or None when they are not a png, jpeg, gif or webp image."
    if not isinstance(b, (bytes, bytearray)): return None
    if b[:4] == b'RIFF' and b[8:12] == b'WEBP': return 'image/webp'
    return first(m for sig, m in IMAGE_TYPES.items() if b.startswith(sig))

The first bytes give the type. A text file with an image name is not an image.

In [ ]:
assert image_type(b'\x89PNG\r\n\x1a\n') == 'image/png' and image_type(b'\xff\xd8\xff\xe0') == 'image/jpeg' and image_type(b'GIF89a..') == 'image/gif'
assert image_type(b'RIFF\x00\x00\x00\x00WEBPVP8 ') == 'image/webp' and image_type(b'<svg/>') is None and image_type(None) is None

The host gives a resolver that turns an image id into a file. `asset_resolver` gives the file only when it is inside the assets
folder of the vault, exists, and has an image suffix.

In [ ]:
#| export
def asset_resolver(lr:Learning) -> Callable[[str], Path|None]:
    "A resolver from an image id `asset:<path>` to a file in the assets folder of `lr`, or None."
    root = lr.vault.assets().resolve()
    def resolve(image_id:str) -> Path|None:
        if not _image_ok(image_id): return None
        p = (root/image_id.split(':', 1)[1]).resolve()
        return p if p.is_relative_to(root) and p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES else None
    return resolve

`_figure` reads one figure through the resolver. A figure that the resolver does not find, that is too large, or that is not an
image gives None. A resolver that raises gives None too.

In [ ]:
#| export
def _figure(resolve, f) -> tuple|None:
    "The sha256, the bytes and the type of the figure `f`, or None."
    try: p = resolve(f) if f else None
    except Exception: p = None
    if p is None: return None
    try:
        p = Path(p)
        if p.stat().st_size > val('talks.image_mb') * 2**20: return None
        b = p.read_bytes()
    except OSError: return None
    return (hashlib.sha256(b).hexdigest(), b, m) if (m := image_type(b)) else None

A slide shows its passage when it has the beat `passage`, or when it has no beats.

In [ ]:
#| export
def _passage(s:dict) -> str:
    "The passage that slide `s` shows to the room, or ''."
    beats = s.get('beats') or []
    return ((s.get('show') or {}).get('passage') or '') if 'passage' in beats or not beats else ''

`public_deck` gives the title, the public slides and the images by hash. Each slide has exactly the fields `id`, `kind`, `title`,
`section`, `passage`, `figure` and `ask`.

In [ ]:
#| export
def public_deck(t:dict, resolve) -> AttrDict:
    "The part of talk `t` that the room can see; `resolve` turns an image id into a file."
    slides, images = [], {}
    for s in t['slides']:
        fig = _figure(resolve, (s.get('show') or {}).get('figure'))
        if fig: images[fig[0]] = fig[1:]
        slides.append(dict(id=s['id'], kind=s['kind'], title=s.get('title') or '', section=s.get('section') or '', passage=_passage(s),
                           figure=fig[0] if fig else None, ask=deepcopy(public_ask(s.get('ask')))))
    return AttrDict(title=t['title'], slides=slides, images=images)

## The run log

Each time the presenter gives the talk, a run log keeps what happened: the place, the time on each slide, the rounds of each
question, the hands, the questions from the room that the presenter kept, and the primers that were opened beside a slide.
It keeps counts only. It keeps no device token, no address and no answer of one person.

`new_run` starts a run on the first idea slide. `key` is a random 64-bit value. The channel name of the two windows has it, so
an old window never drives a new run.

In [ ]:
#| export
def new_run(t:dict, now:float=None) -> dict:
    "A new run log of talk `t`, on its first idea slide."
    now = time.time() if now is None else now
    sid = first(s['id'] for s in t['slides'] if s.get('kind') == 'idea') or first(s['id'] for s in t['slides'])
    return dict(id=f'run-{secrets.token_hex(4)}', talk=t['id'], key=secrets.token_hex(8), started=now, ended=None, people=None,
                at=dict(slide=sid, beat=1, blank=False, pane=None, ask=None), shown=[dict(slide=sid, at=now, secs=0.)],
                asks={}, questions=[], beside=[])

`log_go` moves the place. It gives the time since the last move to the last slide that was shown. A new slide gets a new entry.
The presenter can go back, so one slide can have more than one entry. `end_run` closes the last entry and ends the run.

In [ ]:
#| export
def _close(run:dict, now:float):
    "Give the last shown slide its time up to `now`."
    last = run['shown'][-1]; last['secs'] = max(0., now - last['at'])

def log_go(run:dict, slide:str, beat:int, now:float):
    "Move the place of `run` to beat `beat` of `slide`."
    _close(run, now)
    if run['shown'][-1]['slide'] != slide: run['shown'].append(dict(slide=slide, at=now, secs=0.))
    run['at'].update(slide=slide, beat=beat)

def end_run(run:dict, now:float=None) -> dict:
    "End `run` at `now`."
    now = time.time() if now is None else now
    _close(run, now); run['ended'] = now
    return run

A question has one or two rounds. `counts` has one count for each option of a choice, and `[right, not right]` for an order.
A words round has no counts. For hands, there are no rounds: `log_main` keeps the option that most hands chose.
A board or a timeline gives a list: the cards that the room chose, or the order.

In [ ]:
#| export
def _ask_log(run:dict, slide:str, kind:str) -> dict:
    "The log of the question of `slide` in `run`, made when it is not there."
    return run['asks'].setdefault(slide, dict(kind=kind, rounds=[], main=None, shown=[]))

def log_round(run:dict, slide:str, kind:str, counts:list, n:int, **extra) -> dict:
    "Add a round of the question of `slide`, with `n` answers. A third round raises `ValueError`."
    a = _ask_log(run, slide, kind)
    if len(a['rounds']) >= 2: raise ValueError('a question has at most two rounds')
    r = dict(n=n, **extra) if counts is None else dict(counts=list(counts), n=n, **extra)
    a['rounds'].append(r)
    return r

def log_main(run:dict, slide:str, kind:str, main:int|list):
    "Keep `main`, the option that most hands chose for the question of `slide`, or a list (the cards of a board, the order of a timeline)."
    _ask_log(run, slide, kind)['main'] = list(main) if isinstance(main, (list, tuple)) else main

`log_question` keeps a question from the room, with the note of the presenter. `log_beside` keeps a primer that was open beside a slide.

In [ ]:
#| export
def log_question(run:dict, text:str, at:float, shown:bool=False, note:str='') -> dict:
    "Add a question from the room to `run`; its id is `q1`, `q2` ..."
    q = dict(id=f"q{len(run['questions']) + 1}", text=text, at=at, shown=shown, note=note)
    run['questions'].append(q)
    return q

def log_beside(run:dict, slug:str, slide:str, at:float, secs:float):
    "Keep the primer `slug` that was open beside `slide` for `secs` seconds."
    run['beside'].append(dict(slug=slug, slide=slide, at=at, secs=secs))

A run is a document of kind `talk-run`. `runs` gives the runs of one talk, the newest first.

In [ ]:
#| export
@patch
def put_run(self:Learning, r:dict) -> AttrDict:
    "Save run `r`, and replace an earlier one with the same id."
    r = {k: v for k, v in dict(r).items() if k != 'doc_id'}
    d = self.v.add(f"Talk run: {r['talk']} ({r['id']})", title=f"Talk run {r['id']}", source=_src('run', r['id']), kind='talk-run', meta=r)
    if not d.get('changed'): self.v.set_meta(d['doc_id'], **r)
    return self.run(r['id'])

@patch
def run(self:Learning, rid:str) -> AttrDict|None:
    "One run by id."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('run', rid)]))
    return _tmeta(r) if r else None

@patch
def runs(self:Learning, tid:str) -> L:
    "The runs of talk `tid`, the newest first."
    rs = self._docs('talk-run', "json_extract(meta, '$.talk')=?", [tid])
    return L(sorted(map(_tmeta, rs), key=lambda r: -(r.get('started') or 0)))

## After the talk

`share_right` gives the share of right answers in one round, or None for a question with no answer and for a round with no answers.
A vote is split when the share is from `talks.split_lo` to `talks.split_hi`. Then the room talks in pairs and votes again.

In [ ]:
#| export
def share_right(ask:dict, round:dict) -> float|None:
    "The share of right answers of `round` of the question `ask`, or None."
    n, c = (round or {}).get('n') or 0, (round or {}).get('counts')
    if not ask or not n or not isinstance(c, list) or not c: return None
    k, a = ask.get('kind'), ask.get('answer')
    if k == 'choice' and isinstance(a, int) and 0 <= a < len(c): return c[a] / n
    return c[0] / n if k == 'order' else None

def is_split(ask:dict, round:dict, lo:float=None, hi:float=None) -> bool:
    "True when the share right of `round` is from `lo` to `hi` (defaults `talks.split_lo` and `talks.split_hi`)."
    lo, hi = val('talks.split_lo') if lo is None else lo, val('talks.split_hi') if hi is None else hi
    return (s := share_right(ask, round)) is not None and lo <= s <= hi

`weak` gives the slides where the room was unsure: the last round has a share right below `below` (default `talks.weak`), or
most hands chose a wrong option. `top_wrong` is the wrong option that most people chose. For hands it is the option that most
hands chose.

In [ ]:
#| export
def _option(ask:dict, i) -> str|None:
    "The text of option `i` of `ask`, or None."
    opts = ask.get('options') or []
    return opts[i] if isinstance(i, int) and 0 <= i < len(opts) else None

def _top_wrong(ask:dict, counts) -> str|None:
    "The wrong option of a choice with the most answers, or None."
    if ask.get('kind') != 'choice': return None
    wrong = [(c, i) for i, c in enumerate(counts or []) if i != ask.get('answer') and c]
    return _option(ask, max(wrong, key=lambda x: x[0])[1]) if wrong else None

def weak(t:dict, run:dict, below:float=None) -> list:
    "The slides of talk `t` where the room of `run` was unsure, in slide order."
    below, out = val('talks.weak') if below is None else below, []
    for s in t['slides']:
        a, log = s.get('ask'), (run.get('asks') or {}).get(s['id'])
        if not a or not log: continue
        if log.get('rounds'):
            r = log['rounds'][-1]
            if (sh := share_right(a, r)) is not None and sh < below:
                out.append(dict(slide=s['id'], title=s.get('title') or '', share=sh, top_wrong=_top_wrong(a, r.get('counts')), n=r.get('n'), main=None))
        elif isinstance(m := log.get('main'), int) and a.get('kind') == 'choice' and m != a.get('answer'):
            out.append(dict(slide=s['id'], title=s.get('title') or '', share=None, top_wrong=_option(a, m), n=None, main=m))
    return out

`recap` gives the data of the page "After the talk": the unsure slides, each shown slide with its time and share, the questions
from the room, the primers opened beside a slide, and the ideas for a follow-up walk.

In [ ]:
#| export
def _shown(t:dict, run:dict) -> list:
    "Each shown slide of `t` one time, in the order of its first show, with its total seconds."
    secs, by = {}, {s['id']: s for s in t['slides']}
    for e in run.get('shown') or []: secs[e['slide']] = secs.get(e['slide'], 0.) + (e.get('secs') or 0.)
    return [(by[sid], n) for sid, n in secs.items() if sid in by]

def _shares(s:dict, run:dict) -> list:
    "The share right of each round of the question of slide `s`."
    return [share_right(s.get('ask'), r) for r in ((run.get('asks') or {}).get(s['id']) or {}).get('rounds') or []]

def recap(t:dict, run:dict) -> AttrDict:
    "The data of the recap of `run` of talk `t`."
    unsure, by = weak(t, run), {s['id']: s for s in t['slides']}
    slides = [dict(slide=s['id'], title=s.get('title') or '', secs=n, share=(_shares(s, run) or [None])[-1]) for s, n in _shown(t, run)]
    follow = list(dict.fromkeys(c for w in unsure if (c := by[w['slide']].get('concept'))))
    return AttrDict(unsure=unsure, slides=slides, questions=list(run.get('questions') or []), beside=list(run.get('beside') or []), follow=follow)

The recap file is one HTML file for the room. It needs no server: the style is inline, with the token names of the themes and
the light values of Paper, and the figures are `data:` URLs. It has no script, no speaker notes and no names. All text goes
through `html.escape`, so text from the room stays text.

In [ ]:
#| export
RECAP_CSS = """:root { --bg: hsl(44 43% 93%); --fg: hsl(28 22% 17%); --ink2: hsl(30 18% 28%); --mute: hsl(32 20% 34%); --line: hsl(40 30% 80%);
  --acc: hsl(22 42% 33%); --ok: hsl(88 32% 25%); --no: hsl(14 58% 34%); --rule: color-mix(in srgb, var(--fg) 22%, transparent);
  --serif: 'Literata', ui-serif, Georgia, serif; --head: 'Libre Baskerville', ui-serif, Georgia, serif; --mono: ui-monospace, 'SF Mono', Menlo, monospace; }
body { max-width: 44rem; margin: 0 auto; padding: 2rem 1.25rem; background: var(--bg); color: var(--fg); font: 1.05rem/1.6 var(--serif); }
h1, h2 { font-family: var(--head); font-weight: 400; line-height: 1.25; }
h1 { font-size: 2rem; margin: 0; }
h2 { font-size: 1.4rem; margin: 0 0 .25rem; }
section { border-top: 1px solid var(--rule); padding: 1.25rem 0; }
.meta { color: var(--mute); font-size: .9rem; margin: .25rem 0; }
blockquote { margin: .75rem 0; padding-left: 1rem; border-left: 3px solid var(--line); color: var(--ink2); white-space: pre-line; }
img { max-width: 100%; height: auto; }
.q { font-weight: 600; margin-bottom: .25rem; }
.right { color: var(--ok); font-weight: 600; }
.note { color: var(--ink2); }"""

`_ask_html` draws one question: the options with the right one marked, the share right of each round, the hands, and the words
answers that the presenter showed.

In [ ]:
#| export
_LETTERS = 'ABCDEF'

def _ask_html(a:dict, log:dict) -> str:
    "The HTML of the question `a` and its result `log`."
    e, log = html.escape, log or {}
    out, opts, ans = [f'<p class="q">{e(str(a.get("q") or ""))}</p>'], a.get('options') or [], a.get('answer')
    if a['kind'] == 'choice':
        mark = ' <span class="right">✓ Right</span>'
        out.append('<ol type="A">' + ''.join(f'<li>{e(str(o))}{mark if i == ans else ""}</li>' for i, o in enumerate(opts)) + '</ol>')
    elif a['kind'] == 'order' and isinstance(ans, list):
        out.append('<p class="meta">The right order:</p><ol>' + ''.join(f'<li>{e(str(_option(a, i)))}</li>' for i in ans) + '</ol>')
    for k, r in enumerate(log.get('rounds') or [], 1):
        sh = share_right(a, r)
        res = f'{round(100 * sh)}% right, ' if sh is not None else ''
        out.append(f'<p class="meta">Round {k}: {res}{int(r.get("n") or 0)} answers</p>')
    if isinstance(m := log.get('main'), int) and 0 <= m and a['kind'] == 'choice' and m < len(_LETTERS) and isinstance(ans, int) and ans < len(_LETTERS):
        out.append(f'<p class="meta">Most hands: {_LETTERS[m]} (right: {_LETTERS[ans]})</p>')
    if (ws := log.get('shown')): out.append('<ul>' + ''.join(f'<li>{e(str(w))}</li>' for w in ws) + '</ul>')
    return ''.join(out)

`recap_html` gives the file: the title and the date, then each shown slide with its title, passage, figure, question and result,
then the questions from the room that the presenter kept, with the notes. The slides come from the public deck, so the file can
not show more than the room saw, apart from the right answers.

In [ ]:
#| export
def recap_html(t:dict, run:dict, resolve) -> str:
    "The recap of `run` of talk `t` as one HTML file, with the figures inline."
    e, pub = html.escape, public_deck(t, resolve)
    ps = {s['id']: s for s in pub.slides}
    d = datetime.fromtimestamp(run.get('started') or time.time())
    out = [f'<header><h1>{e(str(t["title"]))}</h1><p class="meta">{d:%B} {d.day}, {d.year}</p></header>']
    for s, _ in _shown(t, run):
        p, part = ps[s['id']], [f'<h2>{e(str(ps[s["id"]]["title"]))}</h2>']
        if p['section']: part.append(f'<p class="meta">{e(str(p["section"]))}</p>')
        if p['passage']: part.append(f'<blockquote>{e(str(p["passage"]))}</blockquote>')
        if (h := p['figure']):
            b, m = pub.images[h]
            part.append(f'<img alt="Figure" src="data:{m};base64,{base64.b64encode(b).decode()}">')
        if (a := s.get('ask')) and a.get('kind') != 'none': part.append(_ask_html(a, (run.get('asks') or {}).get(s['id'])))
        out.append(f'<section>{"".join(part)}</section>')
    kept = [q for q in run.get('questions') or [] if q.get('shown')]
    if kept:
        qs = ''.join(f'<li><p>{e(str(q.get("text") or ""))}</p>' + (f'<p class="note">{e(str(q["note"]))}</p>' if q.get('note') else '') + '</li>' for q in kept)
        out.append(f'<section><h2>Questions from the room</h2><ul>{qs}</ul></section>')
    return (f'<!doctype html>\n<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">'
            f'<title>{e(str(t["title"]))}</title><style>{RECAP_CSS}</style></head><body>{"".join(out)}</body></html>\n')

## Tests

These tests use the small paper in `tests/walk_data.py`, the walk of `tests/talk_data.py` and a fake model.

In [ ]:
#| hide
import sys, abhyasa
from pathlib import Path
sys.path.insert(0, str(Path(abhyasa.__file__).parents[1]/'tests'))
from talk_data import *
from abhyasa.store import temp_learning
from abhyasa.llm import Fake

A talk from the walk: one slide for each idea, then the questions and the recap.

In [ ]:
lr = temp_learning(); jid = talk_walk(lr); ask = Fake(TALK_RULES)
t = talk_from_walk(lr, ask, jid)
s1, s2, s3 = t.slides[:3]
assert [s['id'] for s in t.slides] == ['s1', 's2', 's3', 's4', 's5']
assert [s['kind'] for s in t.slides] == ['idea', 'idea', 'idea', 'questions', 'recap']

The first slide has the title, the section, the step and the concept of its idea. It shows the passage, and its question is a choice from the short check.

In [ ]:
assert (s1['title'], s1['section'], s1['step'], s1['concept']) == ('Sequential computation', '1 Introduction', 0, 'sequential-computation')
assert s1['show']['passage'].startswith('This inherently sequential nature') and s1['show']['figure'] is None and s1['beats'] == ['passage']
assert s1['ask'] == dict(kind='choice', **CHOICE, src=dict(check='short'))

The second slide shows the figure as an image id, and its notes are the note of the step. The third slide shows the demo, and its question comes from the gate `hot`.

In [ ]:
assert s2['show']['figure'] == 'asset:attention/images/fig2.png' and s2['beats'] == ['passage', 'figure'] and s2['notes'] == 'Softmax makes weights from scores.'
assert s3['show']['demo'] == 'softmax-temperature' and s3['beats'] == ['passage', 'demo']
assert s3['ask'] == dict(kind='choice', q='T goes from 1 to 2. What happens to the share of Paris?',
                         options=['It rises', 'It falls', 'It stays the same'], answer=1, src=dict(demo='softmax-temperature', gate='hot'))

The minutes add to 20. The demo step makes no model call. The talk is saved, and it passes its checks.

In [ ]:
assert [s['mins'] for s in t.slides] == [4, 4, 5, 5, 2] and talk_minutes(t) == 20 and t.minutes == 20
assert len([c for c in ask.calls if 'You turn one check into a choice question' in c[0] + c[1]]) == 2   # no call for the demo step
assert t.src == {'walk': jid} and t.schema == TALK_SCHEMA and t.live is None and t.phones is False
assert check_talk(t) == [] and lr.talk(t.id).title == 'Attention Is All You Need' and [x.id for x in lr.talks()] == [t.id]

A bad reply, or `choice=False`, gives a words question.

In [ ]:
lr2 = temp_learning(); jid2 = talk_walk(lr2)
bad = talk_from_walk(lr2, Fake({**TALK_RULES, 'You turn one check into a choice question': dict(q='', options=[], answer=9)}), jid2)
assert bad.slides[0]['ask'] == dict(kind='words', q='Say in one or two sentences: what is Sequential computation?', most=80, src=dict(check='short'))
off = talk_from_walk(lr2, Fake(TALK_RULES), jid2, choice=False)
assert off.slides[1]['ask']['kind'] == 'words'

A demo step with no demo falls back to its idea: no demo on the slide, and a choice from a short check.

In [ ]:
lr2.v.forget(lr2.demo('softmax-temperature').doc_id)
nodemo = talk_from_walk(lr2, Fake(TALK_RULES), jid2)
assert nodemo.slides[2]['show']['demo'] is None and nodemo.slides[2]['beats'] == ['passage'] and nodemo.slides[2]['ask']['kind'] == 'choice'

A step with no passage and a figure gives a slide with the figure beat only, and the talk passes its checks.

In [ ]:
j2 = lr2.journey(jid2)
lr2.walk_step(jid2, 1, read={**j2.steps[1]['read'], 'text': ''})
nopass = talk_from_walk(lr2, Fake(TALK_RULES), jid2)
assert nopass.slides[1]['show']['passage'] == '' and nopass.slides[1]['beats'] == ['figure'] and check_talk(nopass) == []

The talk id matches the form of an id, also for a long title. `drop_talk` removes a talk.

In [ ]:
assert all(re.fullmatch(r'[a-z0-9-]{1,48}', x.id) for x in lr2.talks()) and re.fullmatch(r'[a-z0-9-]{1,48}', _talk_id('Ä very long title ' * 9))
lr2.drop_talk(bad.id)
assert lr2.talk(bad.id) is None and bad.id not in [x.id for x in lr2.talks()]

A question from a gate needs fixed options and an answer that the spec holds.

In [ ]:
g = gallery()
assert ask_from_gate(g['parabola-vertex'], 'move') == dict(kind='choice', q='b goes from 0 to 2. Where does the lowest point move?',
                                                          options=['Left and down', 'Right and down', 'Straight up', 'It stays'], answer=0)
assert ask_from_gate(g['reciprocal-rank-fusion'], 'win') is None   # the answer needs derive
assert ask_from_gate(g['vowel-sandhi'], 'join') is None            # the options need derive
assert ask_from_gate(g['softmax-temperature'], 'nope') is None

Options and an answer from a state path give a choice, and the prompt gets the values of the state. A `{name}` with no value stays.

In [ ]:
spec = dict(state=dict(k=60, opts=['x', 'y'], best='y'), blocks=[dict(type='row', blocks=[dict(type='gate', id='g', prompt='With k = {k} and {m}?',
                                                                                         options={'from': 'opts'}, answer='best')])])
assert ask_from_gate(spec, 'g') == dict(kind='choice', q='With k = 60 and {m}?', options=['x', 'y'], answer=1)

The public part of a question has no answer and no source.

In [ ]:
assert public_ask(s3['ask']) == dict(kind='choice', q='T goes from 1 to 2. What happens to the share of Paris?', options=['It rises', 'It falls', 'It stays the same'])
assert public_ask(None) is None

`TALK_CASES`: each talk gives exactly its problems, and `put_talk` refuses a talk with a problem.

In [ ]:
GOOD = dict(id='t-1', kind='talk', schema=1, title='T', src={'walk': 'w-1'}, minutes=10, phones=False, live=None,
            slides=[dict(id='s1', kind='idea', title='A', section='', notes='', mins=3,
                         show=dict(passage='Scores become weights.', figure='asset:a/fig.png', demo=None),
                         beats=['passage', 'lit:weights', 'figure'], ask=dict(kind='choice', q='Q?', options=['a', 'b'], answer=1))])
for case, want in TALK_CASES: assert check_talk(case) == want, (case, want)
wants = [w for _, w in TALK_CASES]
assert [] in wants and ['schema 2 is not 1'] in wants and ["s1: unknown kind 'video'"] in wants
assert ['s1: the answer 2 is not an option'] in wants and ['s1: a choice needs 2 to 6 options'] in wants
assert ["s1: the beat 'demo' needs a demo to show"] in wants and ["s1: the beat 'lit:gradients' lights text that is not in the passage"] in wants
assert ["s1: the figure '/Users/k/fig.png' is not an image id"] in wants and ["s1: the figure 'asset:../vault.db' is not an image id"] in wants
assert ['s1: the slide id is used twice'] in wants and ["s1: unknown ask kind 'poll'"] in wants and ["s1: unknown beat 'fly'"] in wants
assert ['s1: the answer [0, 0, 1] is not an order of the options'] in wants
test_fail(lambda: lr.put_talk(dict(GOOD, schema=2)), contains='schema 2 is not 1')

The good talk of the cases is the same as `GOOD`, and there is one case for each problem.

In [ ]:
assert TALK_CASES[0][0] == GOOD and len(TALK_CASES) == 13

### The public deck, the run log and the recap

A talk from the walk, with the figure file in the assets folder.

In [ ]:
lr = temp_learning(); jid = talk_walk(lr); t = talk_from_walk(lr, Fake(TALK_RULES), jid)
pub, h = public_deck(t, asset_resolver(lr)), hashlib.sha256(PNG).hexdigest()

Each public slide has only the allowed fields. The figure comes by its hash, and the question has no answer.

In [ ]:
assert all(set(s) == {'id', 'kind', 'title', 'section', 'passage', 'figure', 'ask'} for s in pub.slides)
assert pub.slides[1]['figure'] == h and pub.images == {h: (PNG, 'image/png')}
assert pub.slides[2]['ask'] == dict(kind='choice', q='T goes from 1 to 2. What happens to the share of Paris?', options=['It rises', 'It falls', 'It stays the same'])

The slides as JSON have no notes, no answer, no demo and no path.

In [ ]:
js = json.dumps(pub.slides)
assert 'Softmax makes weights' not in js and '"answer"' not in js and 'softmax-temperature' not in js and '/Users/' not in js and '/tmp/' not in js

A slide with beats but no `passage` beat shows no passage to the room.

In [ ]:
t3 = deepcopy(t); t3.slides[0]['beats'] = []
assert public_deck(t3, asset_resolver(lr)).slides[0]['passage'].startswith('This inherently sequential nature')
t3.slides[1]['beats'] = ['figure']
assert public_deck(t3, asset_resolver(lr)).slides[1]['passage'] == ''

A path out of the assets, or a file that is not an image, gives no figure.

In [ ]:
t2 = deepcopy(t); t2.slides[1]['show']['figure'] = 'asset:../vault.db'
assert public_deck(t2, asset_resolver(lr)).slides[1]['figure'] is None
(lr.vault.assets('attention')/'images'/'fake.png').write_bytes(b'<script>alert(1)</script>')
t2.slides[1]['show']['figure'] = 'asset:attention/images/fake.png'
assert public_deck(t2, asset_resolver(lr)).images == {}

A figure larger than `talks.image_mb` does not go to the room, and a resolver that raises gives no figure.

In [ ]:
big = lr.vault.assets('attention')/'images'/'big.png'
big.write_bytes(PNG + bytes(val('talks.image_mb') * 2**20))
t2.slides[1]['show']['figure'] = 'asset:attention/images/big.png'
assert public_deck(t2, asset_resolver(lr)).slides[1]['figure'] is None
def boom(f): raise OSError('no disk')
assert public_deck(t, boom).images == {}

The run log keeps the time on each slide. A move on the same slide adds no entry, and the time goes to the last slide.

In [ ]:
r = new_run(t, now=100.)
assert r['id'].startswith('run-') and len(r['key']) == 16 and r['at'] == dict(slide='s1', beat=1, blank=False, pane=None, ask=None)
log_go(r, 's2', 1, now=160.); log_go(r, 's2', 2, now=170.); log_go(r, 's3', 1, now=400.)
assert r['shown'] == [dict(slide='s1', at=100., secs=60.), dict(slide='s2', at=160., secs=240.), dict(slide='s3', at=400., secs=0.)]
assert r['at']['slide'] == 's3' and r['at']['beat'] == 1

The hands, the rounds, the side pane and the questions. A question has at most two rounds.

In [ ]:
log_main(r, 's1', 'choice', 1)
log_round(r, 's2', 'choice', [20, 2, 1, 1], 24); log_round(r, 's2', 'choice', [22, 1, 1, 0], 24)
test_fail(lambda: log_round(r, 's2', 'choice', [1, 1, 1, 1], 4), contains='at most two rounds')
log_round(r, 's3', 'choice', [11, 10, 4], 25)
log_beside(r, 'softmax', 's3', at=420., secs=120.)
q = log_question(r, 'Is √dk the best number?', at=430., shown=True, note='It keeps the variance near 1.')

`log_main` also keeps a list, as a copy, for a board or a timeline. A single value stays a single value.

In [ ]:
rb, cards = new_run(t, now=0.), (2, 0, 1)
log_main(rb, 's4', 'order', cards); log_main(rb, 's1', 'choice', 1)
assert rb['asks']['s4']['main'] == [2, 0, 1] and rb['asks']['s1']['main'] == 1

`end_run` closes the last slide. The run is saved, and `runs` finds it.

In [ ]:
end_run(r, now=1300.); lr.put_run(r)
assert q['id'] == 'q1' and r['ended'] == 1300. and r['shown'][-1]['secs'] == 900. and lr.runs(t.id)[0]['id'] == r['id']
assert lr.run(r['id']).asks == r['asks'] and lr.run('run-none') is None

A second save replaces the run, and `runs` gives the newest run first.

In [ ]:
r2 = new_run(t, now=2000.); lr.put_run(r2); r2['people'] = 12; lr.put_run(r2)
assert [x['id'] for x in lr.runs(t.id)] == [r2['id'], r['id']] and lr.run(r2['id']).people == 12 and lr.runs('other') == []

Weak ideas: the last round below 0.7, or hands that chose a wrong option.

In [ ]:
w = weak(t, r)
assert [x['slide'] for x in w] == ['s1', 's3']
assert w[1] == dict(slide='s3', title='Scaling by the root of dk', share=.4, top_wrong='It rises', n=25, main=None)
assert (w[0]['share'], w[0]['main'], w[0]['top_wrong']) == (None, 1, 'The largest score only')

A split vote is from 0.3 to 0.7. A round with no answers is not split.

In [ ]:
assert is_split(t.slides[2]['ask'], dict(counts=[11, 10, 4], n=25)) and not is_split(t.slides[1]['ask'], dict(counts=[22, 1, 1, 0], n=24))
assert not is_split(t.slides[2]['ask'], dict(counts=[0, 0, 0], n=0))
assert share_right(dict(kind='order', options=['a', 'b', 'c'], answer=[2, 0, 1]), dict(counts=[3, 1], n=4)) == .75
assert share_right(dict(kind='words', q='Q?', most=80), dict(n=9)) is None and not is_split(dict(kind='words', q='Q?', most=80), dict(n=9))

The recap data: the ideas to follow up, the primers beside a slide, and the questions. The room is not the owner, so the run adds no attempt.

In [ ]:
rc = recap(t, r)
assert rc.follow == ['sequential-computation', 'scaling-by-the-root-of-dk'] and rc.beside == [dict(slug='softmax', slide='s3', at=420., secs=120.)]
assert rc.questions[0]['text'] == 'Is √dk the best number?' and len(lr.attempts(limit=1000)) == 0   # the room is not the owner
assert [(x['slide'], x['secs'], x['share']) for x in rc.slides] == [('s1', 60., None), ('s2', 240., 22/24), ('s3', 900., .4)]

The recap file has no script and no notes. It has the figures inline, the right answers and the shares, and the text from the room is escaped.

In [ ]:
log_question(r, '<img src=x onerror=alert(1)>', at=440., shown=True)
page = recap_html(t, r, asset_resolver(lr))
assert '<script' not in page.lower() and 'src="http' not in page and 'Softmax makes weights' not in page
assert 'data:image/png;base64,' in page and '✓ Right' in page and '40% right' in page and 'It keeps the variance near 1.' in page
assert '<img src=x' not in page and '&lt;img src=x onerror=alert(1)&gt;' in page

The file shows the hands, both rounds, and only the questions that the presenter kept.

In [ ]:
log_question(r, 'A question that the presenter did not keep', at=450.)
page = recap_html(t, r, asset_resolver(lr))
assert 'Most hands: B (right: A)' in page and 'Round 1: 83% right' in page and 'Round 2: 92% right' in page
assert 'did not keep' not in page and page.count('✓ Right') == 3 and '"answer"' not in page

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()